# R2-S3 step 1 - measure the Protocol B selection validation recordings (Colab)

The shortcut probe is trained on B train (measured in R2-S2) and scored on B **selection** validation, so the selection recordings need the same
per-recording measurements. CPU runtime is enough. Run all cells top to bottom; it is safe to rerun after a disconnect.

**Scope:** the 2,000 retained recordings whose validation role is `selection` (genuine, MeloTTS, VITS, XTTS; 25 speakers). Calibration and operating
recordings, test recordings and ChatterboxTTS are never listed or read. The measurement code is the unchanged `eda/measure.py` from R2-S2.

**Produces** in `data/eda/r2_s3_B_selection/` on Drive: `recording_stats_B_selection_v1.parquet` and `r2_s3_measure_manifest.json`, plus the evidence
report `reports/r2/r2_s3_colab_<timestamp>.json`. The probe itself (models, duration matching, memo) runs locally afterwards:
`python scripts/r2_s3_shortcut.py probe`.

In [ ]:
import os, sys, json, subprocess, time, hashlib, platform, shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get("ECHOPROOF_ROOT", "/content/drive/MyDrive/PAI&AIES Project"))
assert ROOT.exists(), f"project folder not found: {ROOT}"

SOURCES = {"__init__eda.py": "src/echoproof/eda/__init__.py", "measure.py": "src/echoproof/eda/measure.py", "selection.py": "src/echoproof/eda/selection.py",
           "shortcut.py": "src/echoproof/eda/shortcut.py", "r2_s1.py": "src/echoproof/features/r2_s1.py", "stats.py": "src/echoproof/features/stats.py",
           "acoustic.py": "src/echoproof/features/acoustic.py", "cohort.py": "src/echoproof/data/cohort.py", "splits.py": "src/echoproof/data/splits.py",
           "chunks.py": "src/echoproof/data/chunks.py", "seeding.py": "src/echoproof/training/seeding.py", "contracts.py": "src/echoproof/contracts.py",
           "bootstrap.py": "src/echoproof/evaluation/bootstrap.py", "metrics.py": "src/echoproof/evaluation/metrics.py",
           "r2_s3_shortcut.py": "scripts/r2_s3_shortcut.py", "test_r2_s3_shortcut.py": "tests/test_r2_s3_shortcut.py",
           "test_r2_s1_features.py": "tests/test_r2_s1_features.py"}
missing = [k for k, rel in SOURCES.items() if not (ROOT / rel).exists()]
assert not missing, f"not found under {ROOT}: {missing} - sync these files to Drive and rerun"
SHA = {k: hashlib.sha256((ROOT / rel).read_bytes()).hexdigest() for k, rel in SOURCES.items()}
print(json.dumps({k: v[:12] for k, v in SHA.items()}, indent=2))

In [ ]:
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "src"))

import numpy, scipy, sklearn, librosa, pyarrow, pandas
ENV = {"python": platform.python_version(), "numpy": numpy.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__,
       "librosa": librosa.__version__, "pyarrow": pyarrow.__version__, "pandas": pandas.__version__, "platform": platform.platform(), "cpu_count": os.cpu_count()}
print(json.dumps(ENV, indent=2))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

INPUTS = {"cohort": ROOT / "data/manifests/cohort_v1.parquet", "split": ROOT / "data/splits/protocol_B_v1.parquet",
          "chunks": ROOT / "data/processed/chunks_v1.parquet", "split_report": ROOT / "reports/r1/split_report.json"}
for name, path in INPUTS.items():
    assert path.exists(), f"missing locked input {name}: {path}"

EXPECTED_COHORT_FILE_SHA256 = "51af27a2dd5c0577bf9c232287ac73b2f66af472ce3308721af83b2e48e8e816"
INPUT_SHA = {name: sha256_file(path) for name, path in INPUTS.items() if name != "split_report"}
print({k: v[:12] for k, v in INPUT_SHA.items()})
assert INPUT_SHA["cohort"] == EXPECTED_COHORT_FILE_SHA256, "the cohort manifest on Drive is not the frozen GR1 file - stop and compare it with the local copy"

import pyarrow.parquet as pq
FROZEN_B = json.loads(INPUTS["split_report"].read_text())["protocols"]["B"]["split_sha256"]
SPLIT_ROWS = [json.loads(r["payload_json"]) for r in pq.read_table(INPUTS["split"]).to_pylist()]
assert {r["split_hash"] for r in SPLIT_ROWS} == {FROZEN_B}, "protocol B split hash differs from the frozen value"
print("protocol B split hash matches the frozen value:", FROZEN_B[:12])

## 1. Unit tests on this runtime (about 2 minutes)

In [ ]:
proc = subprocess.run([sys.executable, "-m", "pytest", "tests/test_r2_s3_shortcut.py", "-q", "-p", "no:cacheprovider"],
                      cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
print(proc.stdout[-3000:] + proc.stderr[-1500:])
UNIT_TESTS_PASSED = proc.returncode == 0
PYTEST_SUMMARY = next((l.strip() for l in reversed(proc.stdout.splitlines()) if " passed" in l or " failed" in l or " error" in l), "no summary line")
print("unit tests:", "PASSED" if UNIT_TESTS_PASSED else "FAILED", "|", PYTEST_SUMMARY)
assert UNIT_TESTS_PASSED, "do not build on a runtime where the tests fail"

## 2. Copy the shards that hold selection recordings from Drive to local disk

In [ ]:
COHORT_ROWS = pq.read_table(INPUTS["cohort"], columns=["recording_id", "canonical_path", "retained", "component", "duration_s", "n_samples", "pcm_sha256"]).to_pylist()
BY_ID = {r["recording_id"]: r for r in COHORT_ROWS}
SELECTION_IDS = [r["recording_id"] for r in SPLIT_ROWS if r["partition"] == "validation" and r["validation_role"] == "selection"]
needed = sorted({BY_ID[i]["canonical_path"].split("::")[0] for i in SELECTION_IDS})
SHARD_SRC, SHARD_DIR = ROOT / "data/processed/cohort_v1", Path("/content/shards")
SHARD_DIR.mkdir(parents=True, exist_ok=True)
print(len(SELECTION_IDS), "selection recordings in", len(needed), "shards:", needed)
t0 = time.perf_counter()
for name in needed:
    src, dst = SHARD_SRC / name, SHARD_DIR / name
    assert src.exists(), f"shard missing on Drive: {src}"
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        continue
    for attempt in range(4):
        try:
            tmp = dst.with_name(dst.name + ".part")
            shutil.copyfile(src, tmp)
            assert tmp.stat().st_size == src.stat().st_size, "size differs after copy"
            os.replace(tmp, dst)
            break
        except OSError as exc:
            print(f"  {name}: attempt {attempt + 1} failed ({exc}); retrying")
            time.sleep(5 * 2 ** attempt)
    else:
        raise RuntimeError(f"could not copy {name}")
    print(f"  copied {name} ({dst.stat().st_size / 1e6:.0f} MB)")
COPY_SECONDS = round(time.perf_counter() - t0, 1)
print("shards ready in", SHARD_DIR, "|", COPY_SECONDS, "s")

## 3. Measure the selection recordings

In [ ]:
from echoproof.eda import measure, selection

OUT_DIR = ROOT / "data/eda/r2_s3_B_selection"
BUILD_SECONDS = None
if (OUT_DIR / selection.MANIFEST_NAME).exists():
    print("already built - loading the existing manifest")
    MANIFEST = json.loads((OUT_DIR / selection.MANIFEST_NAME).read_text())
else:
    t0 = time.perf_counter()
    MANIFEST = selection.build(ROOT, SHARD_DIR, OUT_DIR, seed=42, workers=os.cpu_count() or 1, resume=True, progress=print)
    BUILD_SECONDS = round(time.perf_counter() - t0, 1)
print(json.dumps({k: MANIFEST[k] for k in ("plan_sha256", "scope_sha256", "counts", "seconds_total")}, indent=2))

## 4. Independent cross-checks of the file on Drive

In [ ]:
import numpy as np, pandas as pd
from collections import Counter

problems = []
OUTPUT_HASHES = {}
for key, info in MANIFEST["outputs"].items():
    actual = sha256_file(OUT_DIR / info["file"])
    OUTPUT_HASHES[key] = actual
    if actual != info["sha256"]:
        problems.append(f"{key}: file hash differs from the manifest")

frame = pd.read_parquet(OUT_DIR / selection.OUTPUT_NAMES["stats"])
ids = set(frame["recording_id"])
if ids != set(SELECTION_IDS):
    problems.append("measured recordings differ from the selection assignments in the split file")
if len(frame) != len(SELECTION_IDS):
    problems.append("row count differs from the number of selection assignments")
other = {r["recording_id"] for r in SPLIT_ROWS if r["partition"] != "validation" or r["validation_role"] != "selection"}
if ids & other:
    problems.append("recordings from other roles or partitions appear")
if any(":chatterbox:" in i for i in ids):
    problems.append("Chatterbox recordings appear")
if set(frame["component"]) != {"genuine", "melo", "vits", "xtts"}:
    problems.append(f"components {sorted(set(frame['component']))}")
train_speakers = {r["group_id"] for r in SPLIT_ROWS if r["partition"] == "train"}
if set(frame["group_id"]) & train_speakers:
    problems.append("a selection speaker also appears in train")
dur_gap = max(abs(r.duration_s - BY_ID[r.recording_id]["duration_s"]) for r in frame.itertuples())
len_bad = [r.recording_id for r in frame.itertuples() if r.n_samples != BY_ID[r.recording_id]["n_samples"]]
if dur_gap > 1e-3 or len_bad:
    problems.append(f"durations/lengths differ from the cohort manifest (max gap {dur_gap}, {len(len_bad)} lengths)")
if not np.isfinite(frame[list(measure.SCALAR_COLUMNS)].to_numpy(float)).all():
    problems.append("non-finite measurements")
if MANIFEST["settings"] != measure.SETTINGS:
    problems.append("measurement settings differ from eda/measure.py")

SUMMARY = {"recordings": int(len(frame)), "by_component": dict(Counter(frame["component"])), "speakers": int(frame["speaker_id"].nunique()),
           "median_duration_s": {c: float(frame.loc[frame["component"] == c, "duration_s"].median()) for c in ("genuine", "melo", "vits", "xtts")},
           "median_leading_silence_s": {c: float(frame.loc[frame["component"] == c, "leading_silence_s"].median()) for c in ("genuine", "melo", "vits", "xtts")},
           "max_duration_gap_to_manifest_s": float(dur_gap)}
print(json.dumps(SUMMARY, indent=2))
print("problems:", problems or "none")

## 5. Verdict and evidence

In [ ]:
checks = {"unit_tests_passed": UNIT_TESTS_PASSED, "build_verification_passed": MANIFEST["verification"]["passed"],
          "spot_check_mismatches": MANIFEST["spot_check"]["mismatches"],
          "cohort_manifest_is_frozen_file": INPUT_SHA["cohort"] == EXPECTED_COHORT_FILE_SHA256,
          "split_hash_is_frozen_value": MANIFEST["inputs"]["split_sha256"] == FROZEN_B == MANIFEST["inputs"]["split_sha256_frozen_in_report"],
          "cross_check_problems": problems}
verified = (checks["unit_tests_passed"] and checks["build_verification_passed"] and not checks["spot_check_mismatches"]
            and checks["cohort_manifest_is_frozen_file"] and checks["split_hash_is_frozen_value"] and not problems)
evidence = {"schema_version": 1, "task": "R2-S3", "step": "measure_selection", "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "environment": ENV, "source_sha256": SHA, "input_sha256": INPUT_SHA, "output_sha256": OUTPUT_HASHES,
            "verdict": {"verified": bool(verified), "pytest_summary": PYTEST_SUMMARY, **checks}, "summary": SUMMARY, "build_manifest": MANIFEST,
            "timing": {"shard_copy_seconds": COPY_SECONDS, "build_seconds_this_session": BUILD_SECONDS, "build_seconds_recorded": MANIFEST["seconds_total"]}}
out_dir = ROOT / "reports" / "r2"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / f"r2_s3_colab_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.json"
path.write_text(json.dumps(evidence, indent=2, sort_keys=True, default=str), encoding="utf-8")
print(json.dumps(evidence["verdict"], indent=2, default=str))
print("\nevidence written to:", path)